In [1]:
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [2]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

In [3]:
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join(itos[i] for i in l)

In [5]:
a = encode("Hello World")
decode(a) == "Hello World"

True

In [6]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)

In [8]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [11]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size, ))
    print(ix)
    x = torch.stack([data[i: i + block_size] for i in ix])
    y = torch.stack([data[i + 1: i + 1 + block_size] for i in ix])
    return x, y

xb, yb = get_batch("train")

tensor([ 76049, 234249, 934904, 560986])


In [22]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, input, target):
        # input, target = N x C
        logits = self.token_embedding_table(input)
        N, C, E = logits.shape
        print(f"batch_size: {N}, context length: {C}, emb size: {E}")
        logits = logits.view(N * C, E)
        target = target.view(N * C)
        loss = F.cross_entropy(logits, target)

        return logits, loss

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)

batch_size: 4, context length: 8, emb size: 65
torch.Size([32, 65])


In [23]:
logits = logits[:, -1, :]
logits.shape

IndexError: too many indices for tensor of dimension 2